In [ ]:

# UR Fall corrected fusion leaderboard notebook
# This notebook fixes the label inversion bug.
# Correct mapping:
# raw_label = -1 -> normal -> label 0
# raw_label = 0 or 1 -> fall-related alert -> label 1

!pip -q install numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 scikit-learn==1.5.2 matplotlib==3.8.4 tqdm==4.66.4


In [ ]:

import os, ssl, urllib.request, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import wilcoxon
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score, balanced_accuracy_score
warnings.filterwarnings('ignore')

URFALL_BASE = 'https://fenix.ur.edu.pl/mkepski/ds/data/'
DATA_DIR = Path('/content/urfall_data')
OUT_DIR = Path('/content/ur_fall_corrected_outputs')
TABLE_DIR = OUT_DIR / 'tables'
FIG_DIR = OUT_DIR / 'figures'
for p in [DATA_DIR, OUT_DIR, TABLE_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)
EVENT_CSV = OUT_DIR / 'coobserved_frame_events_corrected.csv'
N_FALL = 30
N_ADL = 40
ACC_WINDOW_MS = 500
TRANSITIONAL_IS_ALERT = True
RANDOM_SEEDS = list(range(10))
print('Output folder:', OUT_DIR)


In [ ]:

def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return True
    try:
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        with urllib.request.urlopen(url, context=ctx, timeout=90) as response:
            data = response.read()
        if len(data) == 0:
            return False
        dest.write_bytes(data)
        return True
    except Exception as error:
        print('failed:', dest.name, error)
        return False

print('Downloading UR Fall files from:', URFALL_BASE)
for filename in ['urfall-cam0-falls.csv', 'urfall-cam0-adls.csv']:
    print(filename, download(URFALL_BASE + filename, DATA_DIR / filename))

count = 0
for kind, n in [('fall', N_FALL), ('adl', N_ADL)]:
    for i in range(1, n + 1):
        for suffix in ['acc', 'data']:
            filename = f'{kind}-{i:02d}-{suffix}.csv'
            if download(URFALL_BASE + filename, DATA_DIR / filename):
                count += 1
print('Downloaded per-sequence files:', count, 'expected:', (N_FALL + N_ADL) * 2)


In [ ]:

def parse_feature_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, header=None)
    df = df.rename(columns={0: 'sequence', 1: 'frame', 2: 'raw_label'})
    feature_cols = [c for c in df.columns if c not in ['sequence', 'frame', 'raw_label']]
    df = df.rename(columns={c: f'cam_feat_{i}' for i, c in enumerate(feature_cols)})
    return df

def norm_seq_id(value):
    text = str(value).strip().lower()
    for kind in ['fall', 'adl']:
        if kind in text:
            after = text.split(kind)[-1]
            digits = ''.join(ch for ch in after if ch.isdigit())
            if digits:
                return f'{kind}-{int(digits):02d}'
    return text

def accel_features(acc_df, center_time_ms, window_ms=ACC_WINDOW_MS):
    lo = center_time_ms - window_ms
    hi = center_time_ms + window_ms
    window = acc_df[(acc_df[0] >= lo) & (acc_df[0] <= hi)]
    if len(window) < 2:
        return None
    sv = window[1].astype(float).values
    ax = window[2].astype(float).values
    az = window[3].astype(float).values
    ay = window[4].astype(float).values
    mag = np.sqrt(ax**2 + ay**2 + az**2)
    def slope(arr):
        if len(arr) < 2:
            return 0.0
        return float(np.polyfit(np.arange(len(arr)), arr, 1)[0])
    return {
        'acc_sv_mean': float(np.mean(sv)),
        'acc_sv_max': float(np.max(sv)),
        'acc_sv_min': float(np.min(sv)),
        'acc_sv_std': float(np.std(sv)),
        'acc_sv_range': float(np.max(sv) - np.min(sv)),
        'acc_sv_slope': slope(sv),
        'acc_mag_mean': float(np.mean(mag)),
        'acc_mag_max': float(np.max(mag)),
        'acc_mag_std': float(np.std(mag)),
        'acc_ax_absmean': float(np.mean(np.abs(ax))),
        'acc_ay_absmean': float(np.mean(np.abs(ay))),
        'acc_az_absmean': float(np.mean(np.abs(az))),
        'acc_ax_std': float(np.std(ax)),
        'acc_ay_std': float(np.std(ay)),
        'acc_az_std': float(np.std(az)),
    }

cam_falls = parse_feature_csv(DATA_DIR / 'urfall-cam0-falls.csv')
cam_adls = parse_feature_csv(DATA_DIR / 'urfall-cam0-adls.csv')
cam_df = pd.concat([cam_falls, cam_adls], ignore_index=True)
cam_df['sequence_id'] = cam_df['sequence'].apply(norm_seq_id)
cam_feat_cols = [c for c in cam_df.columns if c.startswith('cam_feat_')]
print('Camera frames:', len(cam_df))
print('Raw label counts:', cam_df['raw_label'].value_counts().to_dict())
print('Camera feature cols:', len(cam_feat_cols))

rows = []
missing = []
for kind, n in [('fall', N_FALL), ('adl', N_ADL)]:
    for i in range(1, n + 1):
        seq = f'{kind}-{i:02d}'
        acc_path = DATA_DIR / f'{seq}-acc.csv'
        sync_path = DATA_DIR / f'{seq}-data.csv'
        if not acc_path.exists() or not sync_path.exists():
            missing.append(seq)
            continue
        acc_df = pd.read_csv(acc_path, header=None)
        sync_df = pd.read_csv(sync_path, header=None)
        cam_seq = cam_df[cam_df['sequence_id'] == seq]
        if len(cam_seq) == 0:
            missing.append(seq)
            continue
        by_frame = {int(r['frame']): r for _, r in cam_seq.iterrows()}
        for _, sr in sync_df.iterrows():
            frame = int(sr[0])
            time_ms = float(sr[1])
            if frame not in by_frame:
                continue
            cr = by_frame[frame]
            af = accel_features(acc_df, time_ms)
            if af is None:
                continue
            raw_label = int(cr['raw_label'])
            # CORRECTED LABEL MAPPING
            if TRANSITIONAL_IS_ALERT:
                label = 1 if raw_label in [0, 1] else 0
            else:
                label = 1 if raw_label == 1 else 0
            item = {'sequence_id': seq, 'sequence': seq, 'frame': frame, 'time_ms': time_ms, 'raw_label': raw_label, 'label': int(label)}
            for c in cam_feat_cols:
                item[c] = float(cr[c])
            item.update(af)
            rows.append(item)

events = pd.DataFrame(rows)
if events.empty:
    raise RuntimeError('No events built')
acc_feat_cols = [c for c in events.columns if c.startswith('acc_')]
print('Events:', events.shape)
print('Sequences:', events.sequence_id.nunique())
print('Raw label counts:', events.raw_label.value_counts().to_dict())
print('Corrected binary label counts:', events.label.value_counts().to_dict())
print('Corrected alert rate:', events.label.mean())
assert 0.30 < events.label.mean() < 0.40, 'Alert rate is not near expected 0.352. Label mapping may be wrong.'
print('Missing sequences:', missing[:10], 'count:', len(missing))


In [ ]:

def grouped_oof_scores(X, y, groups, n_splits=5):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups)
    splitter = GroupKFold(n_splits=min(n_splits, len(np.unique(groups))))
    scores = np.zeros(len(y), dtype=float)
    for tr, te in splitter.split(X, y, groups):
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight='balanced'))
        model.fit(X[tr], y[tr])
        scores[te] = model.predict_proba(X[te])[:, 1]
    return scores

events['camera_score'] = grouped_oof_scores(events[cam_feat_cols].values, events.label.values, events.sequence_id.values)
events['accel_score'] = grouped_oof_scores(events[acc_feat_cols].values, events.label.values, events.sequence_id.values)
print('Camera AUROC:', roc_auc_score(events.label, events.camera_score))
print('Accel AUROC:', roc_auc_score(events.label, events.accel_score))
print(events.groupby('raw_label')[['label','camera_score','accel_score']].mean())
# Correct sanity check: fall-related labels should have higher risk scores than upright
assert events.groupby('raw_label')['camera_score'].mean().loc[1] > events.groupby('raw_label')['camera_score'].mean().loc[-1]
events.to_csv(EVENT_CSV, index=False)
print('Saved:', EVENT_CSV)
display(events[['sequence_id','frame','raw_label','label','camera_score','accel_score']].head())


In [ ]:

def add_temporal_features(df, score_cols=('camera_score','accel_score'), windows=(3,5,8)):
    df = df.sort_values(['sequence_id','frame']).copy()
    c, h = score_cols
    df['score_mean'] = (df[c] + df[h]) / 2
    df['score_max'] = df[[c,h]].max(axis=1)
    df['score_min'] = df[[c,h]].min(axis=1)
    df['score_product'] = df[c] * df[h]
    df['score_naive_indep'] = 1 - (1 - df[c]) * (1 - df[h])
    df['score_absdiff'] = (df[c] - df[h]).abs()
    for col in [c, h, 'score_product', 'score_mean', 'score_naive_indep']:
        df[f'd_{col}'] = df.groupby('sequence_id')[col].diff().fillna(0)
        for w in windows:
            g = df.groupby('sequence_id')[col]
            df[f'{col}_rollmean_{w}'] = g.transform(lambda s: s.rolling(w, min_periods=1).mean())
            df[f'{col}_rollmax_{w}'] = g.transform(lambda s: s.rolling(w, min_periods=1).max())
            df[f'{col}_rollstd_{w}'] = g.transform(lambda s: s.rolling(w, min_periods=1).std().fillna(0))
    return df.fillna(0)

def metrics_from_scores(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        'f1': f1_score(y, pred, zero_division=0),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y, pred),
        'false_alert_rate': fp/(fp+tn) if (fp+tn)>0 else 0,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
    }

def best_threshold(y, scores, objective='f1'):
    ths = np.linspace(0.01, 0.99, 99)
    best_t, best_v = 0.5, -1e9
    for t in ths:
        pred = (scores >= t).astype(int)
        m = metrics_from_scores(y, pred)
        if objective == 'f1':
            v = m['f1']
        elif objective == 'f1_minus_far':
            v = m['f1'] - 0.5*m['false_alert_rate']
        elif objective == 'precision_safe':
            v = m['precision'] + 0.25*m['f1'] - 0.25*m['false_alert_rate'] if m['recall'] >= 0.60 else -1
        else:
            v = m['f1']
        if v > best_v:
            best_v, best_t = v, t
    return best_t

def stress_scores(df, severity, rng):
    c = df['camera_score'].values.copy()
    h = df['accel_score'].values.copy()
    y = df['label'].values
    # Inflate some non-alerts, weaken some alerts, add noise.
    non = y == 0
    alert = y == 1
    c[non] = np.clip(c[non] + rng.normal(0, 0.10*severity, non.sum()) + rng.binomial(1, 0.08*severity, non.sum())*rng.uniform(0.15,0.45,non.sum()), 0, 1)
    h[non] = np.clip(h[non] + rng.normal(0, 0.10*severity, non.sum()) + rng.binomial(1, 0.10*severity, non.sum())*rng.uniform(0.15,0.45,non.sum()), 0, 1)
    c[alert] = np.clip(c[alert] - rng.binomial(1, 0.10*severity, alert.sum())*rng.uniform(0.05,0.25,alert.sum()) + rng.normal(0,0.05*severity,alert.sum()), 0, 1)
    h[alert] = np.clip(h[alert] - rng.binomial(1, 0.18*severity, alert.sum())*rng.uniform(0.05,0.35,alert.sum()) + rng.normal(0,0.07*severity,alert.sum()), 0, 1)
    out = df.copy()
    out['camera_score'] = c
    out['accel_score'] = h
    return out


In [ ]:

def eval_split(train, test, seed, severity=0.0):
    rng = np.random.default_rng(seed)
    if severity > 0:
        train_eval = stress_scores(train, severity, rng)
        test_eval = stress_scores(test, severity, rng)
    else:
        train_eval, test_eval = train.copy(), test.copy()
    train_tf = add_temporal_features(train_eval)
    test_tf = add_temporal_features(test_eval)
    ytr = train_tf.label.values
    yte = test_tf.label.values
    rows = []
    def add_method(name, train_scores, test_scores, objective='f1'):
        t = best_threshold(ytr, np.asarray(train_scores), objective)
        pred = (np.asarray(test_scores) >= t).astype(int)
        m = metrics_from_scores(yte, pred)
        m.update({'method': name, 'threshold': t, 'seed': seed, 'severity': severity})
        rows.append(m)
    add_method('Camera Only', train_tf.camera_score, test_tf.camera_score)
    add_method('Accelerometer Only', train_tf.accel_score, test_tf.accel_score)
    add_method('Mean Late Fusion', train_tf.score_mean, test_tf.score_mean)
    add_method('Max OR Fusion', train_tf.score_max, test_tf.score_max)
    add_method('Min Agreement Fusion', train_tf.score_min, test_tf.score_min)
    add_method('Product Agreement Fusion', train_tf.score_product, test_tf.score_product)
    add_method('Naive Independence', train_tf.score_naive_indep, test_tf.score_naive_indep)
    add_method('Rule AND Tuned', train_tf.score_min, test_tf.score_min, objective='precision_safe')
    add_method('Rule OR Tuned', train_tf.score_max, test_tf.score_max, objective='f1')
    base_features = ['camera_score','accel_score','score_mean','score_max','score_min','score_product','score_naive_indep','score_absdiff']
    temp_features = [c for c in train_tf.columns if c.startswith(('d_','camera_score_roll','accel_score_roll','score_product_roll','score_mean_roll','score_naive_indep_roll'))]
    model_specs = [
        ('Calibrated Logistic Fusion', LogisticRegression(max_iter=5000, class_weight='balanced'), base_features),
        ('Temporal Logistic Fusion', LogisticRegression(max_iter=5000, class_weight='balanced'), base_features + temp_features),
        ('Temporal Extra Trees', ExtraTreesClassifier(n_estimators=250, random_state=seed, class_weight='balanced', min_samples_leaf=3), base_features + temp_features),
        ('Temporal Random Forest', RandomForestClassifier(n_estimators=250, random_state=seed, class_weight='balanced', min_samples_leaf=3), base_features + temp_features),
        ('Temporal HistGradientBoosting', HistGradientBoostingClassifier(random_state=seed, max_iter=200, learning_rate=0.05), base_features + temp_features),
    ]
    for name, clf, feats in model_specs:
        pipe = make_pipeline(StandardScaler(), clf) if 'Logistic' in name else clf
        pipe.fit(train_tf[feats].values, ytr)
        scores = pipe.predict_proba(test_tf[feats].values)[:,1]
        train_scores = pipe.predict_proba(train_tf[feats].values)[:,1]
        add_method(name, train_scores, scores)
    return rows

all_rows = []
severities = [0.0,0.2,0.4,0.6,0.8,1.0]
groups = events.sequence_id.values
splitter = GroupShuffleSplit(n_splits=len(RANDOM_SEEDS), train_size=0.65, random_state=123)
for seed, (tr, te) in zip(RANDOM_SEEDS, splitter.split(events, events.label, groups)):
    train = events.iloc[tr].reset_index(drop=True)
    test = events.iloc[te].reset_index(drop=True)
    for sev in severities:
        all_rows.extend(eval_split(train, test, seed=seed, severity=sev))
results = pd.DataFrame(all_rows)
results.to_csv(TABLE_DIR/'corrected_leaderboard_all_results.csv', index=False)
print(results.shape)
display(results.head())


In [ ]:

def summarize(df):
    metric_cols = ['f1','precision','recall','balanced_accuracy','false_alert_rate']
    out = df.groupby(['severity','method'])[metric_cols].agg(['mean','std']).reset_index()
    out.columns = ['_'.join(c).strip('_') for c in out.columns]
    return out.sort_values(['severity','f1_mean'], ascending=[True, False])
summary = summarize(results)
summary.to_csv(TABLE_DIR/'corrected_summary.csv', index=False)
clean = summary[summary.severity==0.0].sort_values('f1_mean', ascending=False)
print('Clean leaderboard')
display(clean[['method','f1_mean','precision_mean','recall_mean','balanced_accuracy_mean','false_alert_rate_mean']])
print('Best by stress')
best = summary.loc[summary.groupby('severity')['f1_mean'].idxmax()].sort_values('severity')
display(best[['severity','method','f1_mean','precision_mean','recall_mean','false_alert_rate_mean']])
best.to_csv(TABLE_DIR/'corrected_best_by_stress.csv', index=False)
# Paired Wilcoxon vs camera by severity
pairs=[]
for sev in severities:
    base = results[(results.severity==sev)&(results.method=='Camera Only')].sort_values('seed')
    for method in sorted(results.method.unique()):
        if method=='Camera Only':
            continue
        cur = results[(results.severity==sev)&(results.method==method)].sort_values('seed')
        if len(cur)==len(base)==len(RANDOM_SEEDS):
            for metric in ['f1','false_alert_rate']:
                diff = cur[metric].values - base[metric].values
                try:
                    p = wilcoxon(diff).pvalue if np.any(diff!=0) else 1.0
                except Exception:
                    p = np.nan
                pairs.append({'severity':sev,'method':method,'metric':metric,'mean_delta':diff.mean(),'wins':int((diff>0).sum()),'losses':int((diff<0).sum()),'wilcoxon_p':p})
pairs=pd.DataFrame(pairs)
pairs.to_csv(TABLE_DIR/'corrected_wilcoxon_vs_camera.csv', index=False)
display(pairs.head(20))


In [ ]:

# Figures
plot_methods = ['Camera Only','Accelerometer Only','Naive Independence','Rule AND Tuned','Calibrated Logistic Fusion','Temporal Logistic Fusion','Temporal Extra Trees']
for metric, ylabel, fname in [('f1','F1','corrected_stress_f1.png'),('false_alert_rate','False Alert Rate','corrected_stress_far.png')]:
    plt.figure(figsize=(9,5))
    for method in plot_methods:
        sub = results[results.method==method].groupby('severity')[metric].mean().reset_index()
        if len(sub):
            plt.plot(sub.severity, sub[metric], marker='o', label=method)
    plt.xlabel('Stress severity')
    plt.ylabel(ylabel)
    plt.title(ylabel + ' under corrected UR Fall stress benchmark')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR/fname, dpi=200)
    plt.show()
plt.figure(figsize=(9,5))
clean_plot = clean.head(12).iloc[::-1]
plt.barh(clean_plot.method, clean_plot.f1_mean)
plt.xlabel('Mean clean F1')
plt.title('Corrected clean F1 leaderboard')
plt.tight_layout()
plt.savefig(FIG_DIR/'corrected_clean_f1_leaderboard.png', dpi=200)
plt.show()


In [ ]:

# Create a paper-ready summary and zip
summary_text = []
summary_text.append('# Corrected UR Fall Fusion Leaderboard Summary\n')
summary_text.append('## Label sanity check\n')
summary_text.append(f'Rows: {len(events)}\n')
summary_text.append(f'Sequences: {events.sequence_id.nunique()}\n')
summary_text.append(f'Raw label counts: {events.raw_label.value_counts().to_dict()}\n')
summary_text.append(f'Corrected binary label counts: {events.label.value_counts().to_dict()}\n')
summary_text.append(f'Corrected alert rate: {events.label.mean():.3f}\n')
summary_text.append('\n## Clean leaderboard\n')
summary_text.append(clean[['method','f1_mean','precision_mean','recall_mean','balanced_accuracy_mean','false_alert_rate_mean']].to_markdown(index=False))
summary_text.append('\n\n## Best method by stress severity\n')
summary_text.append(best[['severity','method','f1_mean','precision_mean','recall_mean','false_alert_rate_mean']].to_markdown(index=False))
(OUT_DIR/'corrected_paper_ready_summary.md').write_text('\n'.join(summary_text))
import shutil
zip_path = shutil.make_archive('/content/ur_fall_corrected_outputs', 'zip', OUT_DIR)
print('ZIP saved:', zip_path)
print('Download this:', zip_path)
